# Road Accident Hotspot Analysis – Bengaluru, India
## Presentation 2: Dataset Collection & Understanding + Data Preprocessing & EDA

**Primary dataset:** Bengaluru Traffic Police (BTP) – police-station-wise road crashes, 2018–2025
**Supporting datasets:** Bengaluru city annual crashes 2007–2025, Karnataka district-wise crashes 2020–2023,
MoRTH large-city road accidents 2023–2024.
**Source:** OpenCity – Urban Data Portal (data.opencity.in), licence: Public Domain.

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
FIG = BASE / "outputs" / "figures"
for d in (PROC, FIG):
    d.mkdir(parents=True, exist_ok=True)

# Monochrome "Swiss" style: black / greys + one signal-red accent
INK, MID, LIGHT, ACCENT = "#111111", "#6E6E6E", "#C8C8C8", "#D62828"
sns.set_theme(style="white", font="Arial")
plt.rcParams.update({"axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
                     "grid.color": "#E6E6E6", "grid.linewidth": 0.6})
PALETTE = {"North": ACCENT, "East": INK, "West": MID, "South": LIGHT}
MONO = matplotlib.colors.LinearSegmentedColormap.from_list("mono", ["#FFFFFF", "#BDBDBD", INK])
DIVERGE = matplotlib.colors.LinearSegmentedColormap.from_list("div", [ACCENT, "#FFFFFF", INK])
KEY = {}  # key numbers collected for the presentation


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.close()

## PART 1 – DATASET COLLECTION & UNDERSTANDING
### 1.1 Load every raw file exactly as published

In [2]:
RAW_FILES = {
    "btp_2018_2020": "btp_station_2018_2020.csv",
    "btp_2021_2022": "btp_station_2020_2022.csv",
    "btp_2023": "btp_station_2023.csv",
    "btp_2024": "btp_station_2024.csv",
    "btp_2025": "btp_station_2025.csv",
    "city_annual": "blr_crashes_2007_2025.csv",
    "ka_districts": "ka_district_2020_2023.csv",
    "india_cities": "india_large_cities_2024.csv",
}
# utf-8-sig strips the invisible BOM character some files start with
raw = {k: pd.read_csv(RAW / f, encoding="utf-8-sig", dtype=str) for k, f in RAW_FILES.items()}

inventory = pd.DataFrame(
    [{"dataset": k, "file": RAW_FILES[k], "rows": df.shape[0], "columns": df.shape[1]} for k, df in raw.items()]
)
print("RAW FILE INVENTORY")
print(inventory.to_string(index=False))
KEY["inventory"] = inventory.to_dict("records")

RAW FILE INVENTORY
      dataset                        file  rows  columns
btp_2018_2020   btp_station_2018_2020.csv    45       17
btp_2021_2022   btp_station_2020_2022.csv    45       12
     btp_2023        btp_station_2023.csv    63        8
     btp_2024        btp_station_2024.csv    65        6
     btp_2025        btp_station_2025.csv    68        6
  city_annual   blr_crashes_2007_2025.csv    19        5
 ka_districts   ka_district_2020_2023.csv    39       10
 india_cities india_large_cities_2024.csv    51       14


### 1.2 Inspect structure: columns, sample rows, dtypes (as loaded)

In [3]:
for k, df in raw.items():
    print(f"\n=== {k}: shape {df.shape}")
    print("Columns:", [repr(c) for c in df.columns])
    print(df.head(3).to_string(index=False))

# Problems visible already: trailing spaces in headers ('2020 - Fatal '),
# numbers stored as text with thousands separators ('7,469'), subtotal/total rows mixed with data rows,
# different column sets per year, and station names that change spelling across years.
hdr_issues = [(k, c) for k, df in raw.items() for c in df.columns if c != c.strip()]
print("\nHeaders with stray whitespace:", hdr_issues)
KEY["header_issues"] = [f"{k}: '{c}'" for k, c in hdr_issues]


=== btp_2018_2020: shape (45, 17)
Columns: ["'Zone'", "'Station'", "'2018 - Fatal'", "'2018 - Killed'", "'2018 - Non-Fatal'", "'2018 - Injured'", "'2018 - Total Cases'", "'2019 - Fatal'", "'2019 - Killed'", "'2019 - Non-Fatal'", "'2019 - Injured'", "'2019 - Total Cases'", "'2020 - Fatal '", "'2020 - Killed'", "'2020 - Non-Fatal'", "'2020 - Injured'", "'2020 - Total Cases '"]
Zone     Station 2018 - Fatal 2018 - Killed 2018 - Non-Fatal 2018 - Injured 2018 - Total Cases 2019 - Fatal 2019 - Killed 2019 - Non-Fatal 2019 - Injured 2019 - Total Cases 2020 - Fatal  2020 - Killed 2020 - Non-Fatal 2020 - Injured 2020 - Total Cases 
East Cubbon Park            2             2               52             53                 54            5             5               46             56                 51             2             2               24             28                  26
East   H.Grounds            2             2               58             51                 60            5        

### 1.3 Reshape the five BTP station files into ONE tidy long table (station × year)

In [4]:
def tidy_wide(df, years, has_subdiv):
    """Convert a 'YYYY - Metric' wide file to long format."""
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    rows = []
    for _, r in df.iterrows():
        for y in years:
            def g(*names):
                for n in names:
                    if n in df.columns:
                        return r[n]
                return np.nan
            rows.append({
                "Zone_reported": r["Zone"],
                "Sub_division": r["Sub-division"] if has_subdiv else np.nan,
                "Station_raw": r["Station"],
                "Year": y,
                "Fatal": g(f"{y} - Fatal", f"{y} - Fatal Cases", f"{y}-Fatal crashes"),
                "Killed": g(f"{y} - Killed", f"{y} - Killed People"),
                "Non_Fatal": g(f"{y} - Non-Fatal", f"{y}-Non Fatal"),
                "Injured": g(f"{y} - Injured", f"{y} - Injured People"),
                "Total": g(f"{y} - Total Cases", f"{y}-Total crashes"),
            })
    return pd.DataFrame(rows)


parts = [
    tidy_wide(raw["btp_2018_2020"], [2018, 2019, 2020], False).assign(Source_file=RAW_FILES["btp_2018_2020"]),
    tidy_wide(raw["btp_2021_2022"], [2021, 2022], False).assign(Source_file=RAW_FILES["btp_2021_2022"]),
    tidy_wide(raw["btp_2023"], [2023], True).assign(Source_file=RAW_FILES["btp_2023"]),
    tidy_wide(raw["btp_2024"], [2024], True).assign(Source_file=RAW_FILES["btp_2024"]),
    tidy_wide(raw["btp_2025"], [2025], True).assign(Source_file=RAW_FILES["btp_2025"]),
]
stations_raw = pd.concat(parts, ignore_index=True)
print("Combined long table (before cleaning):", stations_raw.shape)
print(stations_raw.head())

Combined long table (before cleaning): (421, 10)
  Zone_reported Sub_division  Station_raw  Year Fatal Killed Non_Fatal Injured Total                Source_file
0          East          NaN  Cubbon Park  2018     2      2        52      53    54  btp_station_2018_2020.csv
1          East          NaN  Cubbon Park  2019     5      5        46      56    51  btp_station_2018_2020.csv
2          East          NaN  Cubbon Park  2020     2      2        24      28    26  btp_station_2018_2020.csv
3          East          NaN    H.Grounds  2018     2      2        58      51    60  btp_station_2018_2020.csv
4          East          NaN    H.Grounds  2019     5      5        48      54    53  btp_station_2018_2020.csv


### 1.4 Data dictionary – attributes and data types

In [5]:
data_dictionary = pd.DataFrame([
    ["Zone", "Categorical (nominal)", "BTP traffic zone (East / West / North / South) – 2025 structure"],
    ["Zone_reported", "Categorical (nominal)", "Zone exactly as printed in that year's file (zones were reorganised in 2023)"],
    ["Sub_division", "Categorical (nominal)", "Traffic sub-division (published only from 2023)"],
    ["Station", "Categorical (nominal)", "Traffic police station – the spatial unit for hotspot analysis (harmonised name)"],
    ["Station_raw", "Text", "Station name exactly as published"],
    ["Year", "Integer (ordinal / time)", "Calendar year of the crashes, 2018–2025"],
    ["Fatal", "Integer (count, discrete)", "Number of fatal crashes (≥1 person died)"],
    ["Killed", "Integer (count, discrete)", "Number of persons killed (not published for 2024–2025)"],
    ["Non_Fatal", "Integer (count, discrete)", "Number of non-fatal (injury) crashes"],
    ["Injured", "Integer (count, discrete)", "Number of persons injured (not published for 2024–2025)"],
    ["Total", "Integer (count, discrete)", "Total crashes = Fatal + Non_Fatal"],
    ["Fatality_Rate_pct", "Float (ratio, derived)", "Fatal / Total × 100 – share of crashes that were fatal"],
    ["Severity_Index", "Float (ratio, derived)", "Killed per 100 crashes (2018–2023 only)"],
    ["Source_file", "Text", "Raw file the record came from (lineage)"],
], columns=["Attribute", "Data type", "Description"])
print(data_dictionary.to_string(index=False))

        Attribute                 Data type                                                                      Description
             Zone     Categorical (nominal)                  BTP traffic zone (East / West / North / South) – 2025 structure
    Zone_reported     Categorical (nominal)     Zone exactly as printed in that year's file (zones were reorganised in 2023)
     Sub_division     Categorical (nominal)                                  Traffic sub-division (published only from 2023)
          Station     Categorical (nominal) Traffic police station – the spatial unit for hotspot analysis (harmonised name)
      Station_raw                      Text                                                Station name exactly as published
             Year  Integer (ordinal / time)                                          Calendar year of the crashes, 2018–2025
            Fatal Integer (count, discrete)                                         Number of fatal crashes (≥1 person died)


## PART 2 – DATA PREPROCESSING
### 2.1 Remove aggregate rows (sub-totals / zone totals / grand totals)

In [6]:
is_agg = (
    stations_raw["Station_raw"].fillna("").str.contains("Total", case=False)
    | stations_raw["Zone_reported"].isna()
    | stations_raw["Zone_reported"].str.strip().eq("Total")
)
agg_rows = stations_raw[is_agg]
print(f"Aggregate rows found: {len(agg_rows)} (removed from the record-level table, kept for validation)")
print(agg_rows[["Year", "Zone_reported", "Sub_division", "Station_raw", "Total"]].to_string(index=False))
KEY["agg_rows_removed"] = int(len(agg_rows))
grand_totals = agg_rows[agg_rows["Station_raw"].str.contains("Grand", na=False)].copy()
st = stations_raw[~is_agg].copy()

Aggregate rows found: 50 (removed from the record-level table, kept for validation)
 Year Zone_reported Sub_division Station_raw Total
 2018           NaN          NaN Grand Total  4611
 2019           NaN          NaN Grand Total  4684
 2020           NaN          NaN Grand Total  3236
 2021           NaN          NaN Grand Total  3211
 2022           NaN          NaN Grand Total  3823
 2023           NaN        East        Total   337
 2023           NaN   Whitefield       Total   667
 2023          East          NaN  East Total  1004
 2023           NaN      Central       Total   316
 2023           NaN         West       Total   568
 2023           NaN  Vijayanagar       Total   370
 2023          West          NaN  West Total  1254
 2023           NaN   North East       Total   934
 2023           NaN        North       Total   601
 2023         North          NaN North Total  1535
 2023           NaN   South East       Total   332
 2023           NaN        South       Total   55

### 2.2 Data-type correction (text → numeric, strip whitespace, categories)

In [7]:
NUM = ["Fatal", "Killed", "Non_Fatal", "Injured", "Total"]
print("dtypes BEFORE:\n", st.dtypes.to_string())
for c in ["Zone_reported", "Sub_division", "Station_raw"]:
    st[c] = st[c].str.strip()
for c in NUM:
    st[c] = pd.to_numeric(st[c].str.replace(",", "", regex=False).str.strip(), errors="coerce").astype("Int64")
st["Year"] = st["Year"].astype(int)
grand_totals["Total"] = pd.to_numeric(grand_totals["Total"], errors="coerce")
grand_totals["Fatal"] = pd.to_numeric(grand_totals["Fatal"], errors="coerce")

dtypes BEFORE:
 Zone_reported       str
Sub_division     object
Station_raw         str
Year              int64
Fatal               str
Killed           object
Non_Fatal           str
Injured          object
Total               str
Source_file         str


### 2.3 Harmonise station names (same station, different spellings across years)

In [8]:
NAME_MAP = {
    "Ulsoor": "Halasooru", "K.R.Pura": "K R Puram", "White Field": "Whitefield", "HuliMavu": "Hulimavu",
    "Mico layout": "Micolayout", "K.Swamy Lyt.": "K S Layout", "RT Nagar": "R T Nagar",
    "Yelahanka": "Yalahanka", "Chikjala": "Chikkajala", "Malleswaram": "Malleshwaram",
    "Y.Pura": "Yashawanthapura", "B.Pura": "Bytarayanapura", "Chamarajapet": "Chamarajpet",
    "Int. Aiport": "Int. Airport", "F.Town": "Pulikeshinagar", "Chickpet": "Sheshadripuram/Chickpet",
}
# Low-confidence links are NOT merged (kept separate, documented):
UNMERGED_NOTES = {
    "Devanahalli": "Likely predecessor of 'Int. Airport' (both cover the KIA corridor) – not merged, unverified",
    "Vijayanagar/Magadi Road": "Continuous names – no change needed",
}
st["Station"] = st["Station_raw"].replace(NAME_MAP)
name_changes = (st.loc[st["Station"] != st["Station_raw"], ["Station_raw", "Station"]]
                .drop_duplicates().sort_values("Station").reset_index(drop=True))
print("Station names harmonised:")
print(name_changes.to_string(index=False))
KEY["names_harmonised"] = int(len(name_changes))

# Zone: use the 2025 zone structure for every year so zones are comparable over time
latest_zone = (st.sort_values("Year").groupby("Station")[["Zone_reported", "Sub_division"]].last())
st["Zone"] = st["Station"].map(latest_zone["Zone_reported"])
st["Sub_division"] = st["Sub_division"].fillna(st["Station"].map(latest_zone["Sub_division"]))
moved = st.loc[st["Zone"] != st["Zone_reported"], ["Station", "Zone_reported", "Zone"]].drop_duplicates()
print(f"\nStations whose zone changed in the 2023 reorganisation: {moved['Station'].nunique()}")
print(moved.to_string(index=False))
KEY["zone_moved"] = int(moved["Station"].nunique())

Station names harmonised:
 Station_raw                 Station
      B.Pura          Bytarayanapura
Chamarajapet             Chamarajpet
    Chikjala              Chikkajala
      Ulsoor               Halasooru
    HuliMavu                Hulimavu
 Int. Aiport            Int. Airport
    K.R.Pura               K R Puram
K.Swamy Lyt.              K S Layout
 Malleswaram            Malleshwaram
 Mico layout              Micolayout
      F.Town          Pulikeshinagar
    RT Nagar               R T Nagar
    Chickpet Sheshadripuram/Chickpet
 White Field              Whitefield
   Yelahanka               Yalahanka
      Y.Pura         Yashawanthapura

Stations whose zone changed in the 2023 reorganisation: 22
        Station Zone_reported  Zone
    Cubbon Park          East  West
      H.Grounds          East  West
      S.S.Nagar          East  West
         U.Gate          East  West
    Ashokanagar          East  West
       W.Garden          East  West
        Adugodi          East Sou

### 2.4 Missing values

In [9]:
miss = pd.DataFrame({
    "missing_count": st[NUM + ["Sub_division"]].isna().sum(),
    "missing_pct": (st[NUM + ["Sub_division"]].isna().mean() * 100).round(2),
})
print("Missing values per column (station table):\n", miss)
miss_by_year = st.groupby("Year")[NUM].apply(lambda d: d.isna().sum())
print("\nMissing by year:\n", miss_by_year)
KEY["missing_table"] = miss.reset_index().rename(columns={"index": "column"}).to_dict("records")

plt.figure(figsize=(8, 3.6))
sns.heatmap(st.sort_values("Year")[NUM].isna().astype(int).T, cbar=False, cmap=["#EFEFEF", ACCENT])
yr_pos = st.sort_values("Year").reset_index(drop=True).groupby("Year").apply(lambda d: d.index.min())
plt.xticks(yr_pos.values, yr_pos.index, rotation=0)
plt.title("Missing-value map (red = missing) – station records ordered by year")
savefig("01_missing_heatmap")

# Decision: Killed / Injured for 2024-25 are *structurally* missing (BTP stopped publishing them).
# They are NOT imputed – imputing victim counts would invent data. Analyses that need them use 2018-2023.
# Total is complete; Fatal + Non_Fatal are complete.

Missing values per column (station table):
               missing_count  missing_pct
Fatal                     0         0.00
Killed                  103        27.76
Non_Fatal                 0         0.00
Injured                 103        27.76
Total                     0         0.00
Sub_division              5         1.35

Missing by year:
       Fatal  Killed  Non_Fatal  Injured  Total
Year                                          
2018      0       0          0        0      0
2019      0       0          0        0      0
2020      0       0          0        0      0
2021      0       0          0        0      0
2022      0       0          0        0      0
2023      0       0          0        0      0
2024      0      50          0       50      0
2025      0      53          0       53      0


### 2.5 Duplicate records

In [10]:
exact_dups = st.duplicated().sum()
key_dups = st.duplicated(subset=["Station", "Year"], keep=False)
print("Exact duplicate rows:", exact_dups)
print("Duplicate (Station, Year) keys:", key_dups.sum())
print(st[key_dups].to_string(index=False) if key_dups.any() else "None")
# The two files '2018-2020' and '2020-2022' overlap in name – check 2020 does not appear twice
print("Records per year:\n", st["Year"].value_counts().sort_index().to_string())
KEY["exact_dups"] = int(exact_dups)
KEY["key_dups"] = int(key_dups.sum())
st = st.drop_duplicates(subset=["Station", "Year"], keep="first")

Exact duplicate rows: 0
Duplicate (Station, Year) keys: 0
None
Records per year:
 Year
2018    44
2019    44
2020    44
2021    44
2022    44
2023    48
2024    50
2025    53


### 2.6 Consistency / validity checks

In [11]:
st["chk_total"] = (st["Fatal"] + st["Non_Fatal"]) == st["Total"]
st["chk_killed_ge_fatal"] = st["Killed"].isna() | (st["Killed"] >= st["Fatal"])
bad_total = st[~st["chk_total"]]
bad_killed = st[~st["chk_killed_ge_fatal"]]
print(f"Rows where Fatal + Non_Fatal != Total: {len(bad_total)}")
print(bad_total[["Year", "Station", "Fatal", "Non_Fatal", "Total"]].to_string(index=False))
print(f"Rows where Killed < Fatal (impossible): {len(bad_killed)}")
KEY["bad_total"] = int(len(bad_total))
KEY["bad_killed"] = int(len(bad_killed))
KEY["bad_total_rows"] = bad_total[["Year", "Station", "Fatal", "Non_Fatal", "Total"]].astype(str).to_dict("records")

# Reconcile sum of stations vs published Grand Total vs city-level annual table
city = raw["city_annual"].copy()
for c in city.columns:
    city[c] = pd.to_numeric(city[c].str.replace(",", ""), errors="coerce").astype(int)
city.columns = ["Year", "Fatal", "Killed", "Non_Fatal", "Total"]
recon = (st.groupby("Year")[["Fatal", "Total"]].sum()
         .join(grand_totals.set_index(grand_totals["Year"].astype(int))[["Fatal", "Total"]].add_suffix("_GrandTotalRow"))
         .join(city.set_index("Year")[["Fatal", "Total"]].add_suffix("_CityTable")))
recon["Total_diff_vs_city"] = recon["Total"] - recon["Total_CityTable"]
print("\nReconciliation of totals:\n", recon)
KEY["recon"] = recon.reset_index().astype(int).to_dict("records")

Rows where Fatal + Non_Fatal != Total: 0
Empty DataFrame
Columns: [Year, Station, Fatal, Non_Fatal, Total]
Index: []
Rows where Killed < Fatal (impossible): 0

Reconciliation of totals:
       Fatal  Total  Fatal_GrandTotalRow  Total_GrandTotalRow  Fatal_CityTable  Total_CityTable  Total_diff_vs_city
Year                                                                                                              
2018    663   4611                  663                 4611              661             4611                   0
2019    746   4684                  746                 4684              744             4688                  -4
2020    622   3236                  622                 3236              622             3236                   0
2021    618   3211                  618                 3211              621             3213                  -2
2022    751   3823                  751                 3823              752             3822                   1
2023    

### 2.7 Feature engineering (derived ratios)

In [12]:
st["Fatality_Rate_pct"] = (st["Fatal"] / st["Total"] * 100).round(2)
st["Severity_Index"] = (st["Killed"] / st["Total"] * 100).round(2)
st = st[["Year", "Zone", "Zone_reported", "Sub_division", "Station", "Station_raw",
         "Fatal", "Killed", "Non_Fatal", "Injured", "Total", "Fatality_Rate_pct", "Severity_Index",
         "chk_total", "chk_killed_ge_fatal", "Source_file"]].sort_values(["Year", "Zone", "Station"]).reset_index(drop=True)
print("dtypes AFTER:\n", st.dtypes.to_string())
st.to_csv(PROC / "bengaluru_station_crashes_2018_2025_clean.csv", index=False)
city["Fatality_Rate_pct"] = (city["Fatal"] / city["Total"] * 100).round(2)
city.to_csv(PROC / "bengaluru_city_annual_2007_2025_clean.csv", index=False)
KEY["clean_shape"] = list(st.shape)
KEY["n_stations"] = int(st["Station"].nunique())
KEY["stations_per_year"] = st.groupby("Year")["Station"].nunique().to_dict()

dtypes AFTER:
 Year                     int64
Zone                       str
Zone_reported              str
Sub_division            object
Station                    str
Station_raw                str
Fatal                    Int64
Killed                   Int64
Non_Fatal                Int64
Injured                  Int64
Total                    Int64
Fatality_Rate_pct      Float64
Severity_Index         Float64
chk_total              boolean
chk_killed_ge_fatal    boolean
Source_file                str


### 2.8 Supporting datasets – clean the same way

In [13]:
ka = raw["ka_districts"].copy()
ka = ka[ka["District"].notna() & (ka["District"].str.strip() != "")]
ka = ka[~ka["Sl No"].isna()].drop(columns="Sl No")
ka_long = ka.melt(id_vars="District", var_name="metric", value_name="value")
ka_long["Year"] = ka_long["metric"].str.extract(r"(\d{4})").astype(int)
ka_long["metric"] = ka_long["metric"].str.replace(r"\s*\d{4}", "", regex=True)
ka_long["value"] = pd.to_numeric(ka_long["value"], errors="coerce")
ka_clean = ka_long.pivot_table(index=["District", "Year"], columns="metric", values="value").reset_index()
ka_clean.columns.name = None
ka_clean = ka_clean.rename(columns={"Road Crashes": "Crashes"})
print("Karnataka missing values:\n", ka_clean[ka_clean.isna().any(axis=1)])
# Vijayanagara district was carved out of Ballari in 2021 -> 2020/2021 legitimately empty
ka_clean = ka_clean.dropna(subset=["Crashes"])
ka_clean["Deaths_per_100_crashes"] = (ka_clean["Deaths"] / ka_clean["Crashes"] * 100).round(1)
ka_clean.to_csv(PROC / "karnataka_districts_2020_2023_clean.csv", index=False)

ind = raw["india_cities"].copy()
ind = ind[ind["City"].notna()]
for c in ind.columns.drop("City"):
    ind[c] = pd.to_numeric(ind[c].str.replace(",", ""), errors="coerce")
ind = ind[~ind["City"].str.contains("Total", case=False)]
ind.to_csv(PROC / "india_large_cities_2023_2024_clean.csv", index=False)
print("India cities shape:", ind.shape)

Karnataka missing values:
 Empty DataFrame
Columns: [District, Year, Deaths, Crashes]
Index: []


India cities shape: (50, 14)


### 2.9 Outlier detection – IQR and Z-score on station totals

In [14]:
def iqr_flags(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr), q1 - 1.5 * iqr, q3 + 1.5 * iqr


out_rows = []
for y, g in st.groupby("Year"):
    flags, lo, hi = iqr_flags(g["Total"].astype(float))
    z = np.abs(stats.zscore(g["Total"].astype(float)))
    for idx in g.index:
        if flags[idx] or z[g.index.get_loc(idx)] > 2.5:
            out_rows.append({"Year": y, "Station": g.at[idx, "Station"], "Zone": g.at[idx, "Zone"],
                             "Total": int(g.at[idx, "Total"]), "IQR_upper_fence": round(hi, 1),
                             "Z_score": round(float(z[g.index.get_loc(idx)]), 2),
                             "IQR_outlier": bool(flags[idx]), "Z_outlier(>2.5)": bool(z[g.index.get_loc(idx)] > 2.5)})
outliers = pd.DataFrame(out_rows)
print("Outlier station-years:\n", outliers.to_string(index=False))
KEY["outliers"] = outliers.to_dict("records")
KEY["outlier_station_counts"] = outliers["Station"].value_counts().to_dict()
outliers.to_csv(PROC / "outliers_station_year.csv", index=False)

plt.figure(figsize=(11, 4.6))
sns.boxplot(data=st, x="Year", y="Total", color="#F2F2F2", linecolor=INK, fliersize=0)
iqr_out = outliers[outliers["IQR_outlier"]]
is_out = st.set_index(["Year", "Station"]).index.isin(iqr_out.set_index(["Year", "Station"]).index)
sns.stripplot(data=st[~is_out], x="Year", y="Total", hue="Zone", hue_order=list(PALETTE), palette=PALETTE,
              size=4, jitter=0.2, alpha=.9, edgecolor=INK, linewidth=0.3)
plt.scatter(iqr_out["Year"] - 2018, iqr_out["Total"], s=110, facecolors="none", edgecolors=ACCENT,
            lw=2, label="IQR outlier (red ring)", zorder=5)
plt.scatter(iqr_out["Year"] - 2018, iqr_out["Total"], s=22, c=iqr_out["Zone"].map(PALETTE), edgecolors=INK, lw=0.5, zorder=6)
for y, g in st.groupby("Year"):  # label only the highest station each year to keep the chart readable
    r = g.loc[g["Total"].astype(float).idxmax()]
    plt.annotate(r["Station"], (y - 2018, r["Total"]), fontsize=7.5, xytext=(0, 7), textcoords="offset points", ha="center")
plt.ylim(0, 320)
plt.title("Crashes per police station by year – red-ringed points are IQR outliers (highest station labelled)")
plt.ylabel("Total crashes")
plt.legend(title="Zone", fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
savefig("02_boxplot_outliers")

# Temporal anomalies: a station-year far below that station's own median (possible reporting gap)
med = st.groupby("Station")["Total"].transform("median").astype(float)
st_ratio = st["Total"].astype(float) / med
temporal = st.loc[(st_ratio < 0.45) & (med >= 60), ["Year", "Station", "Total"]].assign(
    Station_median=med[(st_ratio < 0.45) & (med >= 60)], Ratio_to_median=st_ratio[(st_ratio < 0.45) & (med >= 60)].round(2))
print("Temporal anomalies (year < 45% of station's own median):\n", temporal.to_string(index=False))
KEY["temporal_anomalies"] = temporal.to_dict("records")
temporal.to_csv(PROC / "temporal_anomalies.csv", index=False)

# City-level time series outliers (year-on-year)
city["Non_Fatal_z"] = stats.zscore(city["Non_Fatal"]).round(2)
city["YoY_change_pct"] = (city["Total"].pct_change() * 100).round(1)
print("\nCity annual series with z-scores and YoY change:\n", city.to_string(index=False))
KEY["city"] = city.to_dict("records")

# Decision: outliers are REAL high-crash stations (they are the hotspots we want to find),
# not data-entry errors -> retained. 2016 city spike (+55% non-fatal) and 2020-21 COVID dip are flagged.

Outlier station-years:


  Year       Station  Zone  Total  IQR_upper_fence  Z_score  IQR_outlier  Z_outlier(>2.5)
 2018     K R Puram  East    283            297.1     2.86        False             True
 2019     K R Puram  East    269            249.8     2.59         True             True
 2019     Yalahanka North    260            249.8     2.45         True            False
 2023     K R Puram  East    254            204.9     2.25         True            False
 2023    Whitefield  East    223            204.9     1.78         True            False
 2023  Int. Airport North    232            204.9     1.92         True            False
 2023     Yalahanka North    236            204.9     1.98         True            False
 2023 Kamakshipalya  West    241            204.9     2.05         True            False
 2023       Kengeri  West    279            204.9     2.62         True             True
 2024     K R Puram  East    241            232.4     2.57         True             True
 2024  Int. Airport 

Temporal anomalies (year < 45% of station's own median):


  Year     Station  Total  Station_median  Ratio_to_median
 2021 Indiranagar     32            79.5             0.40
 2021   K R Puram     64           247.5             0.26

City annual series with z-scores and YoY change:
  Year  Fatal  Killed  Non_Fatal  Total  Fatality_Rate_pct  Non_Fatal_z  YoY_change_pct
 2007    957     981       7469   8426              11.36         2.08             NaN
 2008    864     892       6908   7772              11.12         1.66            -7.8
 2009    737     761       6138   6875              10.72         1.09           -11.5
 2010    816     858       5667   6483              12.59         0.74            -5.7
 2011    727     762       5297   6024              12.07         0.47            -7.1
 2012    740     760       4767   5507              13.44         0.08            -8.6
 2013    737     771       4493   5230              14.09        -0.12            -5.0
 2014    711     737       4293   5004              14.21        -0.27        

## PART 3 – EXPLORATORY DATA ANALYSIS & DESCRIPTIVE STATISTICS
### 3.1 Descriptive statistics

In [15]:
desc = st[["Fatal", "Killed", "Non_Fatal", "Injured", "Total", "Fatality_Rate_pct", "Severity_Index"]].astype(float).describe().T
desc["median"] = st[desc.index].astype(float).median()
desc["skew"] = st[desc.index].astype(float).skew()
desc["kurtosis"] = st[desc.index].astype(float).kurt()
desc["CV_pct"] = desc["std"] / desc["mean"] * 100
desc = desc.round(2)
print("Descriptive statistics – all station-years:\n", desc)
KEY["desc"] = desc.reset_index().rename(columns={"index": "variable"}).to_dict("records")
desc.to_csv(PROC / "descriptive_statistics.csv")

by_year = st.groupby("Year").agg(Stations=("Station", "nunique"), Total=("Total", "sum"), Fatal=("Fatal", "sum"),
                                  Mean_per_station=("Total", "mean"), Median_per_station=("Total", "median"),
                                  Std=("Total", "std"), Max=("Total", "max")).round(1)
by_year["Fatality_Rate_pct"] = (by_year["Fatal"] / by_year["Total"] * 100).round(2)
print("\nBy year:\n", by_year)
KEY["by_year"] = by_year.reset_index().to_dict("records")

by_zone = st.groupby(["Zone", "Year"])[["Total", "Fatal"]].sum().reset_index()
zone_summary = st[st["Year"] >= 2023].groupby("Zone").agg(Total=("Total", "sum"), Fatal=("Fatal", "sum"),
                                                          Stations=("Station", "nunique"),
                                                          Station_years=("Station", "size")).reset_index()
zone_summary["Fatality_Rate_pct"] = (zone_summary["Fatal"] / zone_summary["Total"] * 100).round(2)
zone_summary["Crashes_per_station_per_yr"] = (zone_summary["Total"] / zone_summary["Station_years"]).round(1)
print("\nZone summary 2023–2025:\n", zone_summary)
KEY["zone_summary"] = zone_summary.to_dict("records")
zone_year = by_zone.pivot(index="Zone", columns="Year", values="Total").astype(int)
(BASE / "outputs").mkdir(exist_ok=True)
with open(BASE / "outputs" / "zone_year.json", "w") as f:
    json.dump({z: {str(y): int(v) for y, v in r.items()} for z, r in zone_year.iterrows()}, f, indent=1)

Descriptive statistics – all station-years:
                    count   mean    std   min    25%    50%     75%     max  median  skew  kurtosis  CV_pct
Fatal              371.0  16.01  12.76  0.00   6.00  12.00   22.00   60.00   12.00  1.24      1.09   79.67
Killed             268.0  16.54  13.43  1.00   6.00  12.00   23.25   63.00   12.00  1.21      0.87   81.19
Non_Fatal          371.0  76.25  48.46  5.00  40.00  64.00  103.00  241.00   64.00  1.00      0.56   63.56
Injured            268.0  79.76  55.01  4.00  40.00  63.00  110.25  287.00   63.00  1.09      0.68   68.97
Total              371.0  92.26  59.39  7.00  48.00  77.00  124.50  283.00   77.00  1.01      0.44   64.37
Fatality_Rate_pct  371.0  17.00   7.29  0.00  11.82  16.60   21.12   52.63   16.60  0.92      2.94   42.90
Severity_Index     268.0  17.75   7.54  2.86  12.94  17.01   21.78   52.63   17.01  0.82      1.99   42.46



By year:


       Stations  Total  Fatal  Mean_per_station  Median_per_station   Std  Max  Fatality_Rate_pct
Year                                                                                            
2018        44   4611    663             104.8                90.0  63.0  283              14.38
2019        44   4684    746             106.5                87.0  63.5  269              15.93
2020        44   3236    622              73.5                59.0  49.8  195              19.22
2021        44   3211    618              73.0                64.0  47.8  189              19.25
2022        44   3823    751              86.9                71.0  56.1  205              19.64
2023        48   4974    882             103.6                89.5  67.6  279              17.73
2024        50   4800    852              96.0                82.0  56.9  242              17.75
2025        53   4890    807              92.3                77.0  60.7  267               16.5

Zone summary 2023–2025:
    

### 3.2 Long-term trend – Bengaluru 2007–2025

In [16]:
fig, ax1 = plt.subplots(figsize=(10, 4.2))
ax1.bar(city["Year"], city["Non_Fatal"], color=LIGHT, label="Non-fatal crashes")
ax1.bar(city["Year"], city["Fatal"], bottom=city["Non_Fatal"], color=INK, label="Fatal crashes")
ax1.set_ylabel("Crashes")
ax2 = ax1.twinx()
ax2.plot(city["Year"], city["Fatality_Rate_pct"], color=ACCENT, marker="o", lw=2.4, label="Fatality rate %")
ax2.set_ylabel("Fatal / total (%)")
ax2.grid(False)
ax1.axvspan(2019.5, 2021.5, color="#EEEEEE", alpha=1, zorder=0)
ax1.text(2020.5, 3500, "COVID", ha="center", fontsize=9)
ax1.annotate("2016 spike", (2016, 7506), xytext=(2016.6, 8400), arrowprops=dict(arrowstyle="->"), fontsize=9)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=8, frameon=False)
plt.title("Bengaluru road crashes 2007–2025: totals fall, but fatality share rises")
plt.xticks(city["Year"], rotation=45)
savefig("03_city_trend")
KEY["city_fr_2007"] = float(city.loc[city.Year == 2007, "Fatality_Rate_pct"].iloc[0])
KEY["city_fr_2025"] = float(city.loc[city.Year == 2025, "Fatality_Rate_pct"].iloc[0])

### 3.3 Distributions

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
sns.histplot(st["Total"].astype(float), bins=20, kde=True, ax=axes[0], color=INK)
axes[0].set_title(f"Total crashes / station-year\nskew = {st['Total'].astype(float).skew():.2f}")
sns.histplot(st["Fatal"].astype(float), bins=20, kde=True, ax=axes[1], color=MID)
axes[1].set_title(f"Fatal crashes / station-year\nskew = {st['Fatal'].astype(float).skew():.2f}")
sns.histplot(st["Fatality_Rate_pct"].astype(float), bins=20, kde=True, ax=axes[2], color=ACCENT)
axes[2].set_title(f"Fatality rate % / station-year\nskew = {st['Fatality_Rate_pct'].astype(float).skew():.2f}")
savefig("04_distributions")

### 3.4 Zone comparison

In [18]:
plt.figure(figsize=(9, 4))
sns.lineplot(data=by_zone, x="Year", y="Total", hue="Zone", palette=PALETTE, marker="o", lw=2.4, markeredgecolor=INK)
plt.title("Total crashes by zone (zones mapped to 2025 boundaries)")
plt.legend(title="Zone", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.ylabel("Total crashes")
savefig("05_zone_trend")

### 3.5 Preliminary hotspots – top stations

In [19]:
recent = st[st["Year"] >= 2023].groupby(["Station", "Zone"]).agg(
    Total_3yr=("Total", "sum"), Fatal_3yr=("Fatal", "sum")).reset_index()
recent["Fatality_Rate_pct"] = (recent["Fatal_3yr"] / recent["Total_3yr"] * 100).round(1)
recent["Rank_Total"] = recent["Total_3yr"].rank(ascending=False, method="min").astype(int)
recent["Rank_Fatal"] = recent["Fatal_3yr"].rank(ascending=False, method="min").astype(int)
top = recent.sort_values("Total_3yr", ascending=False).head(12)
print("Top 12 stations 2023–2025:\n", top.to_string(index=False))
KEY["top_stations"] = top.to_dict("records")
recent.sort_values("Total_3yr", ascending=False).to_csv(PROC / "station_ranking_2023_2025.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
t = top.sort_values("Total_3yr")
ax.barh(t["Station"], t["Total_3yr"] - t["Fatal_3yr"], color=LIGHT, label="Non-fatal")
ax.barh(t["Station"], t["Fatal_3yr"], left=t["Total_3yr"] - t["Fatal_3yr"], color=INK, label="Fatal")
for i, (v, f) in enumerate(zip(t["Total_3yr"], t["Fatal_3yr"])):
    ax.text(v + 5, i, f"{v}  ({f} fatal)", va="center", fontsize=8)
ax.set_xlabel("Crashes 2023–2025 (3 years combined)")
ax.set_title("Top-12 crash stations, 2023–2025")
ax.legend(loc="lower right")
savefig("06_top_stations")

# Heatmap: stations present in all 8 years
pivot = st.pivot_table(index="Station", columns="Year", values="Total").astype(float)
full = pivot.dropna()
full = full.loc[full.sum(axis=1).sort_values(ascending=False).index].head(20)
plt.figure(figsize=(9, 7))
sns.heatmap(full, annot=True, fmt=".0f", cmap=MONO, linewidths=0.4, linecolor="white", cbar_kws={"label": "Crashes"}, annot_kws={"size": 7})
plt.title("Top-20 stations (reported every year) – crashes by year")
savefig("07_station_heatmap")
KEY["stations_all_years"] = int(len(pivot.dropna()))

# Rank stability – are hotspots persistent?
rank = pivot.dropna().rank(ascending=False)
rho = rank.corr(method="spearman")
print("Spearman rank correlation between years (stations present all years):\n", rho.round(2))
KEY["rank_rho_2018_2025"] = round(float(rho.loc[2018, 2025]), 2)
KEY["rank_rho_mean_adjacent"] = round(float(np.mean([rho.loc[y, y + 1] for y in range(2018, 2025)])), 2)

Top 12 stations 2023–2025:
         Station  Zone  Total_3yr  Fatal_3yr  Fatality_Rate_pct  Rank_Total  Rank_Fatal
      K R Puram  East        762        109               14.3           1           7
      Yalahanka North        735        168               22.9           2           1
        Kengeri  West        656        115               17.5           3           5
   Int. Airport North        618        161               26.1           4           2
  Kamakshipalya  West        578         81               14.0           5           8
Electronic City South        552        120               21.7           6           3
     Whitefield  East        507        114               22.5           7           6
         Peenya North        496         77               15.5           8           9
     Chikkajala North        491        116               23.6           9           4
        Hebbala North        485         77               15.9          10           9
 Bytarayanapura

Spearman rank correlation between years (stations present all years):
 Year  2018  2019  2020  2021  2022  2023  2024  2025
Year                                                
2018  1.00  0.95  0.95  0.88  0.91  0.90  0.81  0.80
2019  0.95  1.00  0.94  0.84  0.87  0.89  0.84  0.83
2020  0.95  0.94  1.00  0.92  0.93  0.88  0.83  0.83
2021  0.88  0.84  0.92  1.00  0.88  0.84  0.77  0.76
2022  0.91  0.87  0.93  0.88  1.00  0.91  0.82  0.81
2023  0.90  0.89  0.88  0.84  0.91  1.00  0.93  0.88
2024  0.81  0.84  0.83  0.77  0.82  0.93  1.00  0.91
2025  0.80  0.83  0.83  0.76  0.81  0.88  0.91  1.00


### 3.6 Correlation analysis

In [20]:
corr_df = st[st["Year"] <= 2023][["Fatal", "Killed", "Non_Fatal", "Injured", "Total", "Fatality_Rate_pct"]].astype(float)
corr = corr_df.corr().round(2)
print("Pearson correlation (2018–2023, all variables available):\n", corr)
KEY["corr_fatal_total"] = float(corr.loc["Fatal", "Total"])
KEY["corr_fr_total"] = float(corr.loc["Fatality_Rate_pct", "Total"])
plt.figure(figsize=(6.4, 5))
sns.heatmap(corr, annot=True, cmap=DIVERGE, vmin=-1, vmax=1, square=True, linewidths=0.6, linecolor="white")
plt.title("Correlation matrix (2018–2023)")
savefig("08_correlation")

# Fatal vs total scatter – which stations are disproportionately deadly?
plt.figure(figsize=(8, 5))
r = recent
sns.scatterplot(data=r, x="Total_3yr", y="Fatal_3yr", hue="Zone", palette=PALETTE, s=60, edgecolor=INK, linewidth=0.5)
m, b = np.polyfit(r["Total_3yr"], r["Fatal_3yr"], 1)
xs = np.linspace(r["Total_3yr"].min(), r["Total_3yr"].max(), 50)
plt.plot(xs, m * xs + b, "--", color=MID, lw=1)
resid = r["Fatal_3yr"] - (m * r["Total_3yr"] + b)
lab_idx = set(resid.abs().sort_values(ascending=False).head(6).index) | set(r["Total_3yr"].nlargest(6).index)
for _, rr in r.loc[sorted(lab_idx)].iterrows():
    plt.annotate(rr["Station"], (rr["Total_3yr"], rr["Fatal_3yr"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
plt.xlabel("Total crashes 2023–2025")
plt.ylabel("Fatal crashes 2023–2025")
plt.title("Fatal vs total crashes per station, 2023–2025 (dashed = linear fit)")
savefig("09_fatal_vs_total")
KEY["deadly_above_trend"] = r.loc[resid.sort_values(ascending=False).head(5).index, "Station"].tolist()

Pearson correlation (2018–2023, all variables available):
                    Fatal  Killed  Non_Fatal  Injured  Total  Fatality_Rate_pct
Fatal               1.00    1.00       0.82     0.85   0.89               0.41
Killed              1.00    1.00       0.82     0.84   0.88               0.41
Non_Fatal           0.82    0.82       1.00     0.97   0.99              -0.04
Injured             0.85    0.84       0.97     1.00   0.97               0.02
Total               0.89    0.88       0.99     0.97   1.00               0.05
Fatality_Rate_pct   0.41    0.41      -0.04     0.02   0.05               1.00


### 3.7 Context – Karnataka and India

In [21]:
ka23 = ka_clean[ka_clean["Year"] == 2023].sort_values("Crashes", ascending=False)
KEY["ka_blr_share_2023"] = round(float(ka23.loc[ka23.District == "Bengaluru City", "Crashes"].iloc[0] / ka23["Crashes"].sum() * 100), 1)
KEY["ka_top"] = ka23.head(8)[["District", "Crashes", "Deaths", "Deaths_per_100_crashes"]].to_dict("records")
plt.figure(figsize=(9, 4.5))
k = ka23.head(12).sort_values("Crashes")
plt.barh(k["District"], k["Crashes"], color=[ACCENT if d == "Bengaluru City" else LIGHT for d in k["District"]])
plt.title("Karnataka – top-12 police districts by road crashes, 2023")
plt.xlabel("Crashes")
savefig("10_karnataka")

top_cities = ind.sort_values("2024 Accidents", ascending=False).head(10)
KEY["india_top"] = top_cities[["City", "2024 Accidents", "2024 Killed", "2024 Ranking Accidents"]].to_dict("records")
plt.figure(figsize=(9, 4.5))
tc = top_cities.sort_values("2024 Accidents")
plt.barh(tc["City"], tc["2024 Accidents"], color=[ACCENT if c == "Bengaluru" else LIGHT for c in tc["City"]])
plt.title("India – top-10 large cities by road accidents, 2024 (MoRTH)")
plt.xlabel("Accidents")
savefig("11_india_cities")

### 3.8 Save key numbers for the report

In [22]:
def _conv(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (pd.Timestamp,)): return str(o)
    return str(o)

(BASE / "outputs").mkdir(exist_ok=True)
with open(BASE / "outputs" / "key_numbers.json", "w") as f:
    json.dump(json.loads(json.dumps(KEY, default=_conv).replace("NaN", "null")), f, indent=2)
data_dictionary.to_csv(PROC / "data_dictionary.csv", index=False)
print("Saved cleaned data, figures and key numbers.")

Saved cleaned data, figures and key numbers.
